# ಪಾಠ 09 - ಮೆಟಾಕಾಗ್ನಿಷನ್ ವಿನ್ಯಾಸ ಮಾದರಿ


## ಸೆಟಪ್

ಈ ನೋಟ್ಬುಕ್ Microsoft Agent Framework ಅನ್ನು ಬಳಸಿಕೊಂಡು Metacognition ವಿನ್ಯಾಸ ಮಾದರಿಯನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.

**ಮುಂಚಿತ ಅಗತ್ಯತೆಗಳು:**
- ಪರಿಸರ ಚರಗಳನ್ನು ಬಳಸಿಕೊಂಡು ಸಂರಚಿಸಲಾದ Azure OpenAI ನಿಯೋಜನೆ
- Azure CLI ಪ್ರಾಮಾಣೀಕರಿಸಲಾಗಿದೆ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಮೆಟಾಕಾಗ್ನಿಷನ್ ಎಂದರೆ ಏನು?

ಮೆಟಾಕಾಗ್ನಿಷನ್ ಎಂದರೆ **ಆಲೋಚನೆಯ ಬಗ್ಗೆ ಆಲೋಚನೆ**. ಕೃತಕ ಬುದ್ಧಿಮತ್ತೆ ಏಜೆಂಟುಗಳ ಸಂದರ್ಭದಲ್ಲಿ, ಇದರ ಅರ್ಥ ಕೆಳಗಿನ ಕಾರ್ಯಗಳನ್ನು ಮಾಡಲು ಸಾಧ್ಯವಾಗುವ ಏಜೆಂಟುಗಳನ್ನು ನಿರ್ಮಿಸುವುದು:

- **ಸ್ವ-ಪರಿಶೀಲನೆ** ತಮ್ಮ ಸ್ವದ ಫಲಿತಾಂಶಗಳು ಮತ್ತು ತರ್ಕದ ಪ್ರಕ್ರಿಯೆಯ ಮೇಲೆ
- **ದೋಷಗಳನ್ನು ಪತ್ತೆಹಚ್ಚುವುದು** ಮತ್ತು ಮೌನವಾಗಿ ವಿಫಲವಾಗುವುದಕ್ಕಿಂತ ಸೌಮ್ಯವಾಗಿ ಪುನರುದ್ಧಾರಗೊಳ್ಳುವುದು
- **ಮೌಲ್ಯಮಾಪನ** ಅವರ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಸಂಪೂರ್ಣ ಮತ್ತು ಸಹಾಯಕವಾಗಿದ್ದವೆಯೇ ಎಂಬುದನ್ನು
- **ಅನುಗುಣಗೊಳ್ಳುವುದು** ಅವರು ಆರಂಭಿಕ ದೃಷ್ಠಿಕೋನ ಕೆಲಸಮಾಡದಿರುವಾಗ ತಮ್ಮ ರಣನೀತಿಯನ್ನು ಬದಲಿಸಿಕೊಳ್ಳುವುದು (ಉದಾಹರಣೆಗೆ, ಬ್ಯಾಕ್‌ಅಪ್ ವ್ಯವಸ್ಥೆಗೆ ಹಿಂತಿರುಗುವುದು)

ಮೆಟಾಕಾಗ್ನಿಷನ್ ಏಜೆಂಟ್ ಕೇವಲ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸುವುದಲ್ಲ — ಅದು ತನ್ನ ಸ್ವಂತ ಕಾರ್ಯಕ್ಷಮತೆಯನ್ನು ಮೇಲ್ವಿಚಾರಣೆ ಮಾಡಿ, ತಕ್ಷಣವೇ ತಕ್ಕಂತೆ ಹೊಂದಿಕೊಳ್ಳುತ್ತದೆ.


## ಪ್ರಾಥಮಿಕ ಮತ್ತು ಬ್ಯಾಕಪ್ ಉಪಕರಣಗಳು

ಒಂದು ಸಾಮಾನ್ಯ ಮೆಟಾಕಾಗ್ನಿಷನ್ ಮಾದರಿ ಎಂದರೆ **ಫಾಲ್ಬ್ಯಾಕ್ ತಂತ್ರ**. ಏಜೆಂಟ್ ಮೊದಲು ಪ್ರಾಥಮಿಕ ಉಪಕರಣವನ್ನು ಪ್ರಯತ್ನಿಸುತ್ತದೆ; ಅದು ವಿಫಲವಾದರೆ (ಉದಾಹರಣೆಗೆ, 404 ದೋಷ), ಏಜೆಂಟ್ ಆ ವಿಫಲತೆಯನ್ನು ಗುರುತಿಸಿ ಸ್ಪಷ್ಟವಾಗಿ ಬ್ಯಾಕಪ್ ಉಪಕರಣಕ್ಕೆ ಬದಲಾಯಿಸುತ್ತದೆ.

ಇದು ವಾಸ್ತವಿಕ ವ್ಯವಸ್ಥೆಗಳನ್ನು ಪ್ರತಿಬಿಂಬಿಸುತ್ತದೆ, ಅಲ್ಲಿ ಪ್ರಾಥಮಿಕ ಸೇವೆಗಳು ಲಭ್ಯವಿಲ್ಲದಿರಬಹುದು ಮತ್ತು ಏಜೆಂಟ್‌ಗಳು ಪರ್ಯಾಯ ಮಾರ್ಗವನ್ನು ಆಯ್ಕೆಮಾಡುವುದಕ್ಕೂ ಮುನ್ನ ತೊಂದರೆಯನ್ನು ಸ್ವಯಂ ಗುರುತಿಸಬೇಕಾಗುತ್ತದೆ.

ಕೆಳಗೆ ನಾವು ಎರಡು ವಿಮಾನ ಹುಡುಕುವ ಉಪಕರಣಗಳನ್ನು ವಿವರಿಸುತ್ತೇವೆ:
- **ಪ್ರಾಥಮಿಕ** — Paris, Tokyo, ಮತ್ತು Barcelona ಅನ್ನು ಒಳಗೊಂಡಿದೆ
- **ಬ್ಯಾಕಪ್** — Berlin, Sydney, ಮತ್ತು New York City ಅನ್ನು ಒಳಗೊಂಡಿದೆ


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ದೋಷ ಪರಿಹಾರದೊಂದಿಗೆ ಸ್ವ-ಪರಿಶೀಲನೆ ಮಾಡುವ ಏಜೆಂಟ್

ಕೆಳಗಿನ ಏಜೆಂಟ್‌ಗೆ ಪ್ರಾಥಮಿಕ ಫ್ಲೈಟ್ ಸಿಸ್ಟಮ್ ಅನ್ನು ಮೊದಲು ಪ್ರಯತ್ನಿಸಲು, ವೈಫಲ್ಯಗಳನ್ನು ಗುರುತಿಸಲು ಮತ್ತು ಪಾರದರ್ಶಕವಾಗಿ ಬ್ಯಾಕಪ್ ಸಿಸ್ಟಮ್‌ಗೆ ಹಿಂಪಡೆಯಲು ಸೂಚಿಸಲಾಗಿದೆ. ಪ್ರತಿ ಪ್ರತಿಕ್ರಿಯೆಯ ನಂತರ ಅದು ಸಂಕ್ಷಿಪ್ತವಾಗಿ ಸ್ವತಃ ಮೌಲ್ಯಮಾಪನ ಮಾಡುತ್ತದೆ ಎಂಬುದು — ಅದು ಬಳಕೆದಾರರ ಪ್ರಶ್ನೆಯನ್ನು ಸಂಪೂರ್ಣವಾಗಿ ಉತ್ತರಿಸಿತೇ ಎಂಬುದನ್ನು ಪರಿಶೀಲಿಸುತ್ತದೆ.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ಸ್ವ-ಮೌಲ್ಯಮಾಪನ ಮಾದರಿ

ಮೆಟಾಕಾಗ್ನಿಶನ್‌ನ ಮತ್ತೊಂದು ಮುಖಭಾಗವೆಂದರೆ **ಸ್ವ-ಮೌಲ್ಯಮಾಪನ**: ಪ್ರತ್ಯೇಕ ಏಜೆಂಟ್ (ಅಥವಾ ಎರಡನೇ ಓಲಿಯಲ್ಲಿ ಅದೇ ಏಜೆಂಟ್) ಉತ್ತರವನ್ನು ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆಯ ದೃಷ್ಟಿಯಿಂದ ಪರಿಶೀಲಿಸುತ್ತಾನೆ.

ಕೆಳಗೆ ನಾವು `ResponseEvaluator` ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ, ಇದು ಪ್ರವಾಸ-ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಮೂರು ಆಯಾಮಗಳಲ್ಲಿ ಅಂಕನಿರ್ಣಯ ಮಾಡುತ್ತದೆ.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework ಬಳಸಿ **ಮೆಟಾಕಾಗ್ನಿಟಿವ್ ಏಜೆಂಟ್ಗಳನ್ನು** ಹೇಗೆ ರಚಿಸುವುದು ಎಂಬುದನ್ನು ಕಲಿತಿರಿ:

- **ಸ್ವ-ಆಲೋಚನೆ**: ತಮ್ಮದೇ ವಿಚಾರಪ್ರಕ್ರಿಯೆಯನ್ನು ಗಮನಿಸಿ ಏನಾಗಿತೆಂಬುದನ್ನು ಪಾರದರ್ಶಕವಾಗಿ ತಿಳಿಸುವ ಏಜೆಂಟ್‌ಗಳು.
- **Fallbacks ಜೊತೆಗೆ ದೋಷ ಪುನರುದ್ಧಾರ**: ಪ್ರಾಥಮಿಕ + ಬ್ಯಾಕಪ್ ಟೂಲ್ ಮಾದರಿ, ಇಲ್ಲಿ ಏಜೆಂಟ್ ವಿಫಲತೆಯನ್ನು ಕಂಡುಹಿಡಿದು (ಉದಾಹರಣೆಗೆ, 404 ದೋಷಗಳು) ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಬದಲಿ ಮೂಲವನ್ನು ಪ್ರಯತ್ನಿಸುತ್ತದೆ.
- **ಸ್ವ-ಮೌಲ್ಯಮಾಪನ**: ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆಯ ದೃಷ್ಟಿಯಿಂದ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಅಂಕಿತಗೊಳಿಸುವ ಪ್ರತ್ಯೇಕ ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್‌.

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೆಚ್ಚು ದೃಢ, ಪಾರದರ್ಶक ಮತ್ತು ನಂಬಿಸಬಹುದಾದವನ್ನಾಗಿ ಮಾಡುತ್ತವೆ — ಉತ್ಪಾದನಾ ನಿಯೋಜನೆಗಳಿಗಾಗಿ ಅತ್ಯಂತ ಮುಖ್ಯವಾದ ಗುಣಗಳು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ನಿರಾಕರಣೆ:
ಈ ದಾಖಲೆ AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಗಾಗಿ ಪ್ರಯತ್ನಿಸಿದರೂ ಸಹ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅನಿಖರತೆಗಳಿರಬಹುದೆಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಾಖಲೆ ಪ್ರಾಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೆಗಳು ಅಥವಾ ದುರ್ವ್ಯಾಖ್ಯತೆಗಳಿಗಾಗಿ ನಾವು ಜವಾಬ್ದಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
